# Alinhamento e áreas por determinantes

**Módulo 07 — Geometria Analítica Plana e Espacial** · Notebook 02 de 22

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leandrofreiredealmeida/curso_matematica_python/blob/main/07_geometria_analitica_plana_e_espacial/0702a_alinhamento_e_areas_por_determinantes.ipynb)

**Pré-requisitos**
- Módulo 07: [`0701a_plano_cartesiano_ponto_e_distancia.ipynb`](0701a_plano_cartesiano_ponto_e_distancia.ipynb)
- Módulo 03 — Progressões, Matrizes e Sistemas Lineares: [`0305a_determinantes.ipynb`](../03_progressoes_matrizes_e_sistemas_lineares/0305a_determinantes.ipynb) (cálculo e propriedades dos determinantes)
- Módulo 04 — Geometria Plana: [`0411a_poligonos.ipynb`](../04_geometria_plana/0411a_poligonos.ipynb) (divisão em $n - 2$ triângulos), [`0419a_equivalencia_plana.ipynb`](../04_geometria_plana/0419a_equivalencia_plana.ipynb) (mesma base e mesma altura; a "régua de área" do cadarço) e [`0420a_areas_de_superficies_planas.ipynb`](../04_geometria_plana/0420a_areas_de_superficies_planas.ipynb) (área do triângulo pelas coordenadas)

**Objetivos de aprendizagem** — ao final deste notebook você será capaz de:
- calcular a área de um triângulo a partir das coordenadas dos vértices, usando um determinante;
- interpretar o sinal desse determinante como o sentido de percurso (horário ou anti-horário) e usá-lo para saber de que lado de uma reta um ponto está;
- decidir, com uma única conta, se três pontos estão alinhados;
- calcular a área de qualquer polígono pelas coordenadas, com a fórmula do cadarço, e explicar por que ela funciona.

**Tempo estimado:** 80 a 100 minutos

### Antes de começar

Rode a célula abaixo antes de qualquer outra. Ela carrega o `utils_ga.py`.

Uma novidade: as funções que você programou no notebook 01 (`distancia`, `ponto_medio`, `ponto_do_segmento` e `baricentro`) agora também estão guardadas no `utils_ga.py`. Você as deduziu, então pode usá-las à vontade: `uga.distancia(A, B)`, por exemplo. Essa é a regra deste módulo: **uma ferramenta só entra no `utils_ga.py` depois que você entendeu de onde ela vem**.

In [1]:
# Célula de preparação: deixa o notebook pronto para rodar no seu computador ou no Colab.
import sys

if "google.colab" in sys.modules:
    # No Colab, o notebook chega sozinho; baixamos o arquivo de desenhos do repositório.
    import urllib.request
    url = "https://raw.githubusercontent.com/leandrofreiredealmeida/curso_matematica_python/main/07_geometria_analitica_plana_e_espacial/utils_ga.py"
    urllib.request.urlretrieve(url, "utils_ga.py")

import math
import utils_ga as uga
from ipywidgets import interact
import ipywidgets as widgets
import numpy as np

In [2]:
A, B, C = (0, 0), (6, 0), (0, 9)
print(uga.distancia(A, B))        # 6.0
print(uga.ponto_medio(B, C))      # (3.0, 4.5)
print(uga.baricentro(A, B, C))    # (2.0, 3.0)

help(uga.distancia)               # a docstring diz onde ela foi deduzida

6.0
(3.0, 4.5)
(2.0, 3.0)
Help on function distancia in module utils_ga:

distancia(a: 'Ponto', b: 'Ponto') -> 'float'
    Calcula a distância entre os pontos A e B.
    
    Deduzida no notebook 01 (Plano cartesiano, ponto e distância).
    
    É o Teorema de Pitágoras aplicado às diferenças de coordenadas:
    d = √((x_B − x_A)² + (y_B − y_A)²). No espaço, entra também (z_B − z_A)².
    
    Parâmetros:
        a: o ponto A, como (x, y) ou (x, y, z).
        b: o ponto B, com a mesma quantidade de coordenadas.
    
    Retorna:
        A distância, um número maior ou igual a zero.
    
    Levanta:
        ValueError: se os pontos tiverem quantidades diferentes de coordenadas.
    
    Exemplo:
        >>> distancia((1, 2), (4, 6))
        5.0



## Quanto mede este terreno?

Quando um terreno é medido hoje em dia, ninguém sai com uma trena gigante esticando fitas pelo mato. Um técnico caminha até cada **canto** do terreno com um aparelho de GPS e anota as coordenadas. O resultado é uma lista de pontos, como esta (em metros, com a origem num marco escolhido pelo técnico):

| Canto | $x$ (m) | $y$ (m) |
|---|---|---|
| $V_1$ | 0 | 0 |
| $V_2$ | 140 | −10 |
| $V_3$ | 180 | 70 |
| $V_4$ | 120 | 150 |
| $V_5$ | 40 | 130 |
| $V_6$ | −20 | 60 |

E o programa do computador devolve, na hora, a área do terreno.

O terreno é um hexágono torto: não é regular, não tem lados paralelos, não tem ângulos retos. Nenhuma fórmula pronta do módulo 04 se aplica diretamente. Você poderia dividi-lo em triângulos, medir bases e alturas... mas o computador não "vê" o desenho, só tem a tabela.

**Como ele calcula a área usando apenas esses doze números?** No fim deste notebook você vai escrever o programa que faz isso.

In [3]:
# Os cantos do terreno, na ordem em que o técnico caminhou (em metros)
TERRENO = [(0, 0), (140, -10), (180, 70), (120, 150), (40, 130), (-20, 60)]
NOMES_DO_TERRENO = ["V₁", "V₂", "V₃", "V₄", "V₅", "V₆"]
# Onde escrever o nome de cada canto, do lado de fora do terreno
POSICOES_DOS_NOMES = [
    "bottom left", "bottom center", "middle right", "top center", "top center", "middle left",
]


def desenhar_terreno(titulo: str):
    """Desenha o terreno com os cantos e as coordenadas de cada um."""
    fig = uga.criar_plano_cartesiano(
        titulo=titulo, x_min=-40, x_max=200, y_min=-30, y_max=170,
        rotulo_x="x (m)", rotulo_y="y (m)", passo=20,
    )
    fig = uga.desenhar_poligono(fig, TERRENO, cor=uga.CORES["solido"])
    for canto, nome, posicao in zip(TERRENO, NOMES_DO_TERRENO, POSICOES_DOS_NOMES):
        fig = uga.desenhar_ponto(
            fig, canto, rotulo=nome, mostrar_coordenadas=True, posicao_rotulo=posicao
        )
    return fig


uga.mostrar(desenhar_terreno("Terreno medido por GPS: qual é a área?"))

## O que você já sabe e o que muda agora

Este notebook junta duas coisas que você estudou em módulos diferentes: **determinantes** (módulo 03) e **áreas** (módulo 04). Veja o que já está pronto e o que é novo:

| Já visto | Onde | O que muda aqui |
|---|---|---|
| Calcular determinantes de ordem 2 e 3 (Sarrus, Laplace) e suas propriedades | 📎 Módulo 03 — [`0305a_determinantes.ipynb`](../03_progressoes_matrizes_e_sistemas_lineares/0305a_determinantes.ipynb) | Nada de reaprender a calcular. Os determinantes viram uma **ferramenta de geometria**. |
| O determinante 2 × 2 dá a área de um paralelogramo, com um sinal ligado ao sentido horário ou anti-horário | 📎 Módulo 03 — [`0305a_determinantes.ipynb`](../03_progressoes_matrizes_e_sistemas_lineares/0305a_determinantes.ipynb) (seção 1) | Lá, o paralelogramo tinha um canto na origem. Aqui, os triângulos e polígonos podem estar **em qualquer lugar** do plano. E o **sinal**, que lá era uma pergunta de checagem, vira protagonista. |
| A área do triângulo pelas coordenadas, deduzida com o retângulo envolvente | 📎 Módulo 04 — [`0420a_areas_de_superficies_planas.ipynb`](../04_geometria_plana/0420a_areas_de_superficies_planas.ipynb) (seção 3) | Você não vai deduzir a fórmula de novo. Vai descobrir que ela **é um determinante**, e o que isso ensina de novo: o sinal, o alinhamento e a área de **qualquer** polígono. |
| A fórmula do cadarço, usada como "caixa-preta" para conferir áreas | 📎 Módulo 04 — [`0419a_equivalencia_plana.ipynb`](../04_geometria_plana/0419a_equivalencia_plana.ipynb) (seção 1) | **A caixa se abre.** Você vai entender por que ela funciona, até em polígonos com "dentes". |
| Dividir um polígono em triângulos a partir de um vértice e somar as áreas **com sinal** | 📎 Módulo 04 — [`0411a_poligonos.ipynb`](../04_geometria_plana/0411a_poligonos.ipynb) e [`0420a_areas_de_superficies_planas.ipynb`](../04_geometria_plana/0420a_areas_de_superficies_planas.ipynb) (seção 3) | O sinal ganha uma explicação (o sentido de percurso) e o leque passa a sair da **origem**: é assim que a soma vira a fórmula do cadarço. |
| O teste de colinearidade "multiplicando em cruz" e a função `orientacao` (para que lado você vira) | 📎 Módulo 04 — [`0401a_nocoes_e_proposicoes_primitivas.ipynb`](../04_geometria_plana/0401a_nocoes_e_proposicoes_primitivas.ipynb) e [`0408a_quadrilateros_definicao_elementos_e_trapezios.ipynb`](../04_geometria_plana/0408a_quadrilateros_definicao_elementos_e_trapezios.ipynb) | Você vai descobrir que essas contas **eram um determinante** o tempo todo, e o que elas medem. |
| Três pontos alinhados não formam triângulo | 📎 Módulo 07 — [`0701a_plano_cartesiano_ponto_e_distancia.ipynb`](0701a_plano_cartesiano_ponto_e_distancia.ipynb) | No notebook 01 você só conseguia perceber isso olhando (ou comparando distâncias). Agora vai **detectar com uma conta**. |

## 1. Do paralelogramo do módulo 03 ao triângulo em qualquer lugar

### Retomada

No módulo 03 (📎 [`0305a_determinantes.ipynb`](../03_progressoes_matrizes_e_sistemas_lineares/0305a_determinantes.ipynb), seção 1) você viu que, se um paralelogramo tem um canto na origem e os lados saem dela até os pontos $(a, b)$ e $(c, d)$, então a área dele é o valor absoluto do determinante

$$\begin{vmatrix} a & b \\ c & d \end{vmatrix} = ad - bc$$

Um triângulo com um vértice na origem e os outros em $(a, b)$ e $(c, d)$ é **metade** desse paralelogramo (a diagonal o divide em dois triângulos congruentes, 📎 [`0409a_paralelogramos_e_casos_particulares.ipynb`](../04_geometria_plana/0409a_paralelogramos_e_casos_particulares.ipynb)). Então a área dele é $\frac{|ad - bc|}{2}$.

### A novidade: sair da origem

E se nenhum vértice estiver na origem? Use o que você aprendeu no notebook 01: a partir de $A$, olhe para os **deslocamentos** até $B$ e até $C$:

$$(\Delta x_B,\ \Delta y_B) = (x_B - x_A,\ y_B - y_A) \qquad (\Delta x_C,\ \Delta y_C) = (x_C - x_A,\ y_C - y_A)$$

Mover o triângulo inteiro, sem girar, não muda a área. É como se você "arrastasse" o triângulo até $A$ cair na origem: $B$ e $C$ iriam parar exatamente nos pontos $(\Delta x_B, \Delta y_B)$ e $(\Delta x_C, \Delta y_C)$. Então:

> 📌 **Área do triângulo pelos deslocamentos.**
> $$S = \frac{1}{2}\left|\begin{vmatrix} x_B - x_A & y_B - y_A \\ x_C - x_A & y_C - y_A \end{vmatrix}\right|$$

> 📎 **Conexão.** No módulo 03 você chamou esses deslocamentos de **vetores** ($u$ e $v$ do problema do agrimensor). No próximo notebook deste módulo eles ganham um estudo só deles.

🕹️ Use o widget abaixo: ele mostra o triângulo original e a sua cópia "arrastada" até a origem, com o paralelogramo do módulo 03 completando a figura.

In [ ]:
Ponto = tuple[float, float]   # um ponto do plano é uma tupla (x, y), como no notebook 01


def controle(nome: str, valor: int, minimo: int = -6, maximo: int = 6) -> widgets.IntSlider:
    """Cria um slider de números inteiros, que só atualiza a figura ao soltar o botão."""
    return widgets.IntSlider(
        value=valor, min=minimo, max=maximo, step=1, description=nome,
        style={"description_width": "initial"}, continuous_update=False,
    )


def entre_parenteses(numero: float) -> str:
    """Escreve um número negativo entre parênteses, como (−3), para a conta ficar legível."""
    return f"(−{abs(numero):g})" if numero < 0 else f"{numero:g}"


def janela_para(pontos: list, minimo: float = -6, maximo: float = 10) -> tuple:
    """Uma janela que vai de `minimo` a `maximo` e cresce se algum ponto ficar de fora."""
    xs = [p[0] for p in pontos]
    ys = [p[1] for p in pontos]
    return (min(minimo, min(xs) - 1), max(maximo, max(xs) + 1),
            min(minimo, min(ys) - 1), max(maximo, max(ys) + 1))


def mostrar_arrasto(xA: int, yA: int, xB: int, yB: int, xC: int, yC: int) -> None:
    """Desenha o triângulo ABC, a cópia arrastada até a origem e o paralelogramo."""
    A, B, C = (xA, yA), (xB, yB), (xC, yC)

    # A conta: os deslocamentos a partir de A e o determinante 2 × 2 deles
    dxB, dyB = xB - xA, yB - yA
    dxC, dyC = xC - xA, yC - yA
    det = dxB * dyC - dyB * dxC
    S = abs(det) / 2

    # A cópia arrastada: A vai para a origem, B e C vão para os deslocamentos
    O, B_linha, C_linha = (0, 0), (dxB, dyB), (dxC, dyC)
    quarto_vertice = (dxB + dxC, dyB + dyC)   # o canto que completa o paralelogramo

    x_min, x_max, y_min, y_max = janela_para([A, B, C, B_linha, C_linha, quarto_vertice])
    fig = uga.criar_plano_cartesiano(
        titulo="O triângulo ABC arrastado até a origem",
        x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max,
    )
    # O paralelogramo do módulo 03, tracejado
    for P, Q in [(O, B_linha), (B_linha, quarto_vertice), (quarto_vertice, C_linha), (C_linha, O)]:
        fig = uga.desenhar_segmento(
            fig, P, Q, cor=uga.CORES["plano_secundario"], espessura=1.5, tracejado=True
        )
    # O "arrasto" de cada vértice
    for P, P_novo in [(A, O), (B, B_linha), (C, C_linha)]:
        fig = uga.desenhar_seta(fig, P, P_novo, cor=uga.CORES["apoio"], espessura=1)
    fig = uga.desenhar_poligono(
        fig, [O, B_linha, C_linha], rotulos=["O", "B′", "C′"],
        cor=uga.CORES["plano_secundario"],
    )
    fig = uga.desenhar_poligono(fig, [A, B, C], rotulos=["A", "B", "C"])
    uga.mostrar(fig)

    texto_det = f"|{entre_parenteses(det)}|" if det < 0 else f"{det}"
    print(
        f"Deslocamentos: ({dxB}, {dyB}) e ({dxC}, {dyC}) → "
        f"det = {entre_parenteses(dxB)}·{entre_parenteses(dyC)} − "
        f"{entre_parenteses(dyB)}·{entre_parenteses(dxC)} = {det} → "
        f"S = {texto_det}/2 = {S:g}".replace("-", "−")
    )


interact(
    mostrar_arrasto,
    xA=controle("x_A", 2), yA=controle("y_A", 1),
    xB=controle("x_B", 5), yB=controle("y_B", 2),
    xC=controle("x_C", 3), yC=controle("y_C", 5),
);

interactive(children=(IntSlider(value=2, continuous_update=False, description='x_A', max=6, min=-6, style=Slid…

**Pare e pense**

Na fórmula, por que aparece o valor absoluto? O que significaria o determinante dar um número negativo?

<details>
<summary>Ver resposta</summary>

Área não pode ser negativa, então usamos o módulo. Mas o sinal do determinante **não é um erro**: ele carrega uma informação sobre a ordem dos vértices. Guarde essa pergunta: ela é o assunto da seção 3.
</details>

## 2. O determinante dos três pontos

Nos livros e nas provas, a área do triângulo costuma aparecer com um determinante 3 × 3, que tem as coordenadas dos três pontos e uma coluna de 1:

> 📌 **Determinante dos três pontos.**
> $$D = \begin{vmatrix} x_A & y_A & 1 \\ x_B & y_B & 1 \\ x_C & y_C & 1 \end{vmatrix} \qquad\qquad S = \frac{|D|}{2}$$

É a mesma coisa que a fórmula da seção 1, escrita de outro jeito. Você consegue provar isso com as propriedades de determinantes do módulo 03 (📎 [`0305a_determinantes.ipynb`](../03_progressoes_matrizes_e_sistemas_lineares/0305a_determinantes.ipynb), seção 6):

1. **Subtraia a linha 1 da linha 2 e da linha 3.** Somar a uma linha um múltiplo de outra não muda o determinante (é a propriedade da pergunta de checagem da seção 6 do módulo 03; nos livros, ela costuma ser apresentada como consequência do **Teorema de Jacobi**). Subtrair a linha 1 é somar $(-1)$ vezes a linha 1, então:
$$D = \begin{vmatrix} x_A & y_A & 1 \\ x_B - x_A & y_B - y_A & 0 \\ x_C - x_A & y_C - y_A & 0 \end{vmatrix}$$
2. **Desenvolva pela terceira coluna (Laplace).** Só o primeiro elemento dela não é zero, e o cofator dele é $(-1)^{1+3} = +1$ vezes o determinante 2 × 2 que sobra:
$$D = \begin{vmatrix} x_B - x_A & y_B - y_A \\ x_C - x_A & y_C - y_A \end{vmatrix}$$

É exatamente o determinante dos deslocamentos. ∎

E se você **desenvolver** $D$ (por Sarrus ou por Laplace) sem nenhum truque, chega a:

$$D = x_A(y_B - y_C) + x_B(y_C - y_A) + x_C(y_A - y_B)$$

> 📎 **Conexão.** Essa é exatamente a expressão que você deduziu no módulo 04 (📎 [`0420a_areas_de_superficies_planas.ipynb`](../04_geometria_plana/0420a_areas_de_superficies_planas.ipynb), seção 3), com o retângulo envolvente menos três triângulos. Três caminhos diferentes (o retângulo envolvente, os deslocamentos e o determinante 3 × 3) levam à mesma fórmula. Quando isso acontece em matemática, é sinal de que a ideia é importante.

> 💡 **Dica.** A coluna de 1 é só um truque para caber tudo num determinante 3 × 3. Para fazer contas à mão, a forma dos deslocamentos (2 × 2) costuma ser mais rápida.

> ⚠️ **Erro comum.** Esquecer de dividir por 2. O determinante dá a área do **paralelogramo**; o triângulo é a metade.

In [5]:
def determinante_dos_pontos(A: Ponto, B: Ponto, C: Ponto) -> float:
    """D = xA(yB − yC) + xB(yC − yA) + xC(yA − yB): o determinante 3 × 3 desenvolvido."""
    xA, yA = A
    xB, yB = B
    xC, yC = C
    return xA * (yB - yC) + xB * (yC - yA) + xC * (yA - yB)


def determinante_dos_deslocamentos(A: Ponto, B: Ponto, C: Ponto) -> float:
    """O determinante 2 × 2 dos deslocamentos de A até B e de A até C."""
    xA, yA = A
    xB, yB = B
    xC, yC = C
    return (xB - xA) * (yC - yA) - (yB - yA) * (xC - xA)


A, B, C = (1, 2), (5, 3), (2, 7)
D = determinante_dos_pontos(A, B, C)
print("Desenvolvido:      D =", D)
print("Deslocamentos:     D =", determinante_dos_deslocamentos(A, B, C))

# O mesmo determinante 3 × 3, calculado pelo NumPy (com a coluna de 1)
matriz = np.array([[1, 2, 1],
                   [5, 3, 1],
                   [2, 7, 1]])
print("np.linalg.det:     D =", np.linalg.det(matriz))
# O NumPy calcula de outro jeito e às vezes erra na 15ª casa decimal;
# por isso comparamos com math.isclose, e não com ==.
print("Iguais?", math.isclose(np.linalg.det(matriz), D))
print(f"S = |{D}| / 2 = {abs(D) / 2}")

Desenvolvido:      D = 19
Deslocamentos:     D = 19
np.linalg.det:     D = 18.999999999999996
Iguais? True
S = |19| / 2 = 9.5


In [6]:
import random

random.seed(2026)   # semente fixa: a cada execução, os mesmos triângulos são sorteados

concordaram = 0
for _ in range(1000):
    # Três vértices com coordenadas inteiras entre −20 e 20
    A, B, C = [(random.randint(-20, 20), random.randint(-20, 20)) for _ in range(3)]
    caminho_1 = determinante_dos_pontos(A, B, C)
    caminho_2 = determinante_dos_deslocamentos(A, B, C)
    caminho_3 = np.linalg.det(np.array([[A[0], A[1], 1], [B[0], B[1], 1], [C[0], C[1], 1]]))
    if caminho_1 == caminho_2 and math.isclose(caminho_1, caminho_3, abs_tol=1e-9):
        concordaram += 1

print(f"Os três caminhos concordaram em {concordaram} de 1000 triângulos.")

Os três caminhos concordaram em 1000 de 1000 triângulos.


**Pare e pense**

1. Se você trocar a ordem das linhas 2 e 3 no determinante $D$, o que acontece com o valor de $D$? E com a área?
2. Se você somar 10 a todas as abscissas (deslocando o triângulo para a direita), $D$ muda?

<details>
<summary>Ver respostas</summary>

1. Trocar duas linhas troca o sinal do determinante (📎 propriedade 3 da seção 6 de [`0305a_determinantes.ipynb`](../03_progressoes_matrizes_e_sistemas_lineares/0305a_determinantes.ipynb)). A área, que usa o módulo, continua a mesma.
2. Não. Pela forma dos deslocamentos, só as **diferenças** entre coordenadas importam, e elas não mudam quando todos os pontos andam juntos.
</details>

## 3. O sinal de $D$: horário ou anti-horário?

### Retomada

No módulo 03 (📎 [`0305a_determinantes.ipynb`](../03_progressoes_matrizes_e_sistemas_lineares/0305a_determinantes.ipynb), seção 1), uma pergunta de checagem ligou o sinal do determinante ao sentido, horário ou anti-horário, em que se percorre "primeiro $u$, depois $v$". No módulo 04 (📎 [`0408a_quadrilateros_definicao_elementos_e_trapezios.ipynb`](../04_geometria_plana/0408a_quadrilateros_definicao_elementos_e_trapezios.ipynb)), a função `orientacao` usava um sinal parecido para saber se você **vira à esquerda ou à direita** num vértice de um quadrilátero. Agora esse sinal vira uma ferramenta com regra precisa.

Se você percorre os vértices na ordem $A \to B \to C \to A$, você está girando em algum sentido: **anti-horário** (contrário aos ponteiros do relógio) ou **horário**.

> 📌 **Área orientada.** O número $\frac{D}{2}$ é a **área orientada** do triângulo $ABC$:
> - $D > 0$ → o percurso $A \to B \to C$ é **anti-horário**;
> - $D < 0$ → o percurso $A \to B \to C$ é **horário**;
> - $D = 0$ → não há triângulo (seção 4).

Trocar a ordem de dois vértices inverte o sentido do percurso, e por isso troca o sinal de $D$, exatamente como trocar duas linhas de um determinante.

Uma outra forma de ler o sinal, muito usada em programação: **imagine que você está andando de $A$ para $B$**.
- Se $C$ fica à sua **esquerda**, $D > 0$.
- Se $C$ fica à sua **direita**, $D < 0$.

> 📎 **Conexão (uma caixa que se abre).** Lembra da função `orientacao` de [`0408a_quadrilateros_definicao_elementos_e_trapezios.ipynb`](../04_geometria_plana/0408a_quadrilateros_definicao_elementos_e_trapezios.ipynb)? A conta dela era $(x_P - x_O)(y_Q - y_O) - (y_P - y_O)(x_Q - x_O)$: exatamente o **determinante dos deslocamentos** da seção 1. Agora você sabe por que o sinal dela dizia para que lado você vira: ele é o sinal da área orientada.

🕹️ Mova o ponto $C$ e observe a cor do triângulo e o sentido das setas.

In [ ]:
def area_orientada(A: Ponto, B: Ponto, C: Ponto) -> float:
    """Área orientada do triângulo ABC: D/2, com o sinal do percurso A → B → C."""
    return determinante_dos_pontos(A, B, C) / 2


# A cor do triângulo é escolhida AQUI, pelo sinal de D (o uga só desenha)
COR_ANTI_HORARIO = uga.CORES["plano"]
COR_HORARIO = uga.CORES["reta_secundaria"]

A3, B3 = (-3, -2), (4, 1)   # A e B fixos


def mostrar_orientacao(xC: int, yC: int) -> None:
    """Desenha o triângulo ABC colorido pelo sinal de D e o sentido do percurso."""
    C = (xC, yC)
    D = determinante_dos_pontos(A3, B3, C)

    fig = uga.criar_plano_cartesiano(titulo="Andando de A para B: onde está C?")
    fig = uga.desenhar_reta_por_dois_pontos(fig, A3, B3)
    if D > 0:
        fig = uga.desenhar_poligono(fig, [A3, B3, C], cor=COR_ANTI_HORARIO, opacidade=0.3)
        fig = uga.desenhar_sentido_de_percurso(fig, [A3, B3, C])
    elif D < 0:
        fig = uga.desenhar_poligono(fig, [A3, B3, C], cor=COR_HORARIO, opacidade=0.3)
        fig = uga.desenhar_sentido_de_percurso(fig, [A3, B3, C])
    else:
        # Sem triângulo: os três pontos ficam sobre a reta AB
        for P, Q in [(A3, B3), (B3, C), (C, A3)]:
            fig = uga.desenhar_segmento(fig, P, Q, cor=uga.CORES["destaque"])
    # O "caminho de quem anda": de A para B
    fig = uga.desenhar_seta(fig, A3, B3, cor=uga.CORES["reta"], espessura=3)
    fig = uga.desenhar_pontos(fig, [A3, B3], rotulos=["A", "B"], mostrar_coordenadas=True)
    fig = uga.desenhar_ponto(
        fig, C, rotulo="C", cor=uga.CORES["destaque"], mostrar_coordenadas=True
    )
    uga.mostrar(fig)

    area = abs(D) / 2
    if D > 0:
        print(f"D = {D} > 0 → A → B → C é anti-horário; C está à esquerda de quem anda "
              f"de A para B. Área = {area:g}")
    elif D < 0:
        print(f"D = {D} < 0 → A → B → C é horário; C está à direita de quem anda "
              f"de A para B. Área = {area:g}".replace("-", "−"))
    else:
        print("D = 0 → A, B e C estão alinhados: não há triângulo.")


interact(mostrar_orientacao, xC=controle("x_C", 0), yC=controle("y_C", 3));

interactive(children=(IntSlider(value=0, continuous_update=False, description='x_C', max=6, min=-6, style=Slid…

**Pare e pense**

1. No widget, procure duas posições diferentes de $C$ que deem **o mesmo valor** de $D$. Onde elas estão em relação à reta $AB$?
2. No módulo 04 você viu que triângulos com a mesma base e a mesma altura têm a mesma área. Como isso aparece aqui?

<details>
<summary>Ver respostas</summary>

1. Do mesmo lado de $AB$ e à mesma distância dela: numa reta **paralela** a $AB$. Por exemplo, $C(-6, 1)$ e $C(1, 4)$ dão o mesmo $D = 30$. Para ir de um ao outro, você anda 7 para a direita e 3 para cima, exatamente como a reta $AB$ "anda".
2. É a mesma propriedade (📎 [`0419a_equivalencia_plana.ipynb`](../04_geometria_plana/0419a_equivalencia_plana.ipynb), seção 2: triângulos de mesma base e mesma altura são equivalentes), agora vista pelas contas: com $A$ e $B$ fixos, $D$ só depende de quão longe $C$ está da reta $AB$, e de que lado.
</details>

### Para que serve o sinal?

> 🎯 **Aplicação prática.** Saber "de que lado" um ponto está é uma pergunta que computadores fazem o tempo todo:
> - um **carro autônomo** decide se um obstáculo está à esquerda ou à direita da trajetória;
> - um **jogo** decide se o jogador cruzou uma linha de chegada (o sinal muda quando ele passa para o outro lado);
> - um programa de **mapas** decide se você deve virar à esquerda ou à direita na próxima esquina.

E tudo isso com uma conta de multiplicações e subtrações, sem raiz quadrada, sem trigonometria, sem "olhar" para o desenho.

Um exemplo mais interessante: **um ponto $P$ está dentro do triângulo $ABC$?** Pense no que acontece quando você anda pela borda do triângulo, sempre no mesmo sentido: se $P$ está dentro, ele fica **sempre do mesmo lado** de você. Você vai programar esse teste no Exercício Desafio 8.

## 4. Três pontos alinhados

Se $A$, $B$ e $C$ estão sobre uma mesma reta, o "triângulo" fica achatado e a área é zero. E vale o contrário: se a área é zero, os pontos estão alinhados.

> 📌 **Condição de alinhamento.** Os pontos $A$, $B$ e $C$ estão alinhados se, e somente se,
> $$D = \begin{vmatrix} x_A & y_A & 1 \\ x_B & y_B & 1 \\ x_C & y_C & 1 \end{vmatrix} = 0$$

No notebook 01, você só conseguia perceber que três pontos estavam alinhados **olhando a figura** (ou comparando distâncias). Agora basta uma conta.

> 💡 **Dica.** Talvez você pense num outro teste: comparar as "inclinações" de $AB$ e de $AC$, isto é, ver se $\frac{y_B - y_A}{x_B - x_A} = \frac{y_C - y_A}{x_C - x_A}$. Funciona na maioria dos casos, mas tem uma armadilha: se os pontos estiverem numa reta **vertical**, os denominadores valem zero e a conta quebra. O determinante não divide por nada, então nunca quebra. (As inclinações voltam com tudo no notebook 04.)

> 📎 **Conexão (outra caixa que se abre).** Em [`0401a_nocoes_e_proposicoes_primitivas.ipynb`](../04_geometria_plana/0401a_nocoes_e_proposicoes_primitivas.ipynb) você fugiu dessa armadilha "multiplicando em cruz": o teste era $(x_B - x_A)(y_C - y_A) - (y_B - y_A)(x_C - x_A) = 0$. Compare com a seção 1: essa expressão é **exatamente o determinante dos deslocamentos**! Lembra daquela conta que você usou no módulo 04? Agora você sabe o que ela mede: o dobro da área, com sinal, do triângulo $ABC$. Ela dá zero justamente quando o triângulo é achatado.

🕹️ Observe abaixo: $A$ e $B$ estão fixos e $C$ sobe e desce numa reta vertical. Primeiro vem a figura; logo abaixo, o gráfico do valor de $D$ para cada altura de $C$.

In [8]:
import plotly.graph_objects as go

A4, B4 = (-4, -3), (2, 1)   # A e B fixos; C anda na vertical x = 5
ALTURAS_DE_C = [-2 + 0.5 * k for k in range(17)]   # de −2 a 6, de 0,5 em 0,5


def mostrar_alinhamento(yC: float) -> None:
    """Mostra o triângulo ABC e o gráfico de D em função da altura de C."""
    C = (5, yC)
    D = determinante_dos_pontos(A4, B4, C)

    # Figura 1: o triângulo, colorido pelo sinal de D (como na seção 3)
    fig = uga.criar_plano_cartesiano(
        titulo="C anda na reta vertical x = 5", x_min=-6, x_max=8, y_min=-6, y_max=8
    )
    fig = uga.desenhar_segmento(fig, (5, -6), (5, 8), cor=uga.CORES["apoio"], espessura=1)
    fig = uga.desenhar_reta_por_dois_pontos(fig, A4, B4)
    if math.isclose(D, 0, abs_tol=1e-9):
        fig = uga.desenhar_segmento(fig, A4, C, cor=uga.CORES["destaque"])
    else:
        cor = COR_ANTI_HORARIO if D > 0 else COR_HORARIO
        fig = uga.desenhar_poligono(fig, [A4, B4, C], cor=cor, opacidade=0.3)
    fig = uga.desenhar_pontos(fig, [A4, B4], rotulos=["A", "B"], mostrar_coordenadas=True)
    fig = uga.desenhar_ponto(
        fig, C, rotulo="C", cor=uga.CORES["destaque"], mostrar_coordenadas=True
    )
    uga.mostrar(fig)

    # Figura 2: D para cada altura de C (o gráfico de uma conta, sem escala igual)
    valores_de_D = [determinante_dos_pontos(A4, B4, (5, y)) for y in ALTURAS_DE_C]
    grafico = uga.criar_grafico(
        titulo="O valor de D para cada altura de C",
        rotulo_x="altura de C (y_C)", rotulo_y="D", altura=350,
    )
    grafico.add_trace(go.Scatter(
        x=ALTURAS_DE_C, y=valores_de_D, mode="lines", name="D",
        line=dict(color=uga.CORES["plano"], width=3),
        hovertemplate="y_C = %{x}<br>D = %{y}<extra></extra>",
    ))
    grafico.add_hline(y=0, line=dict(color=uga.CORES["destaque"], width=2, dash="dash"))
    grafico.add_trace(go.Scatter(
        x=[yC], y=[D], mode="markers", name="posição atual de C",
        marker=dict(size=13, color=uga.CORES["destaque"]),
        hovertemplate="y_C = %{x}<br>D = %{y}<extra></extra>",
    ))
    uga.mostrar(grafico)

    if math.isclose(D, 0, abs_tol=1e-9):
        print("D = 0 → A, B e C estão alinhados!")
    else:
        print(f"D = {D:g} → não alinhados (área = {abs(D) / 2:g})".replace("-", "−"))


interact(
    mostrar_alinhamento,
    yC=widgets.FloatSlider(
        min=-2, max=6, step=0.5, value=0, description="y_C",
        style={"description_width": "initial"}, continuous_update=False,
    ),
);

interactive(children=(FloatSlider(value=0.0, continuous_update=False, description='y_C', max=6.0, min=-2.0, st…

### Quando $C$ é desconhecido

Repare no gráfico de baixo: $D$ é uma expressão do **1º grau** em $y_C$ (o gráfico é uma reta), e o alinhamento acontece exatamente quando essa expressão vale zero. Isso permite resolver problemas como:

**Exemplo.** Para que valor de $k$ os pontos $A(1, k)$, $B(2, 3)$ e $C(4, 7)$ estão alinhados?

$$D = 1\cdot(3 - 7) + 2\cdot(7 - k) + 4\cdot(k - 3) = -4 + 14 - 2k + 4k - 12 = 2k - 2$$

$D = 0 \iff k = 1$. Os pontos alinhados são $(1, 1)$, $(2, 3)$ e $(4, 7)$.

> 📎 **Conexão.** E se o terceiro ponto for **qualquer** ponto $P(x, y)$? A condição $D = 0$ vira uma equação com $x$ e $y$, e os pontos que a satisfazem são exatamente os pontos da reta $AB$. Essa é uma das formas de obter a **equação da reta**, no notebook 04.

In [9]:
# Promessa do notebook 01 cumprida: agora a função detecta pontos alinhados.

# A função do notebook 01, copiada sem mudanças
def distancia_ao_quadrado(A: Ponto, B: Ponto) -> float:
    """O quadrado da distância entre A e B (a conta da distância, sem a raiz)."""
    xA, yA = A
    xB, yB = B
    return (xB - xA)**2 + (yB - yA)**2


def classificar_triangulo(A: Ponto, B: Ponto, C: Ponto) -> str:
    """Classifica o triângulo ABC pelos lados e pelos ângulos, só com as coordenadas.

    Atenção: a função supõe que A, B e C não estão alinhados. Detectar pontos
    alinhados com uma conta é assunto do notebook 02.
    """
    # Os quadrados dos três lados, do menor para o maior
    q1, q2, q3 = sorted([
        distancia_ao_quadrado(A, B),
        distancia_ao_quadrado(B, C),
        distancia_ao_quadrado(C, A),
    ])

    # Pelos lados: lados iguais têm quadrados iguais
    if math.isclose(q1, q3):
        lados = "equilátero"
    elif math.isclose(q1, q2) or math.isclose(q2, q3):
        lados = "isósceles"
    else:
        lados = "escaleno"

    # Pelos ângulos: recíproca de Pitágoras com o maior lado (q3 é o quadrado dele)
    if math.isclose(q3, q1 + q2):
        angulos = "retângulo"
    elif q3 < q1 + q2:
        angulos = "acutângulo"
    else:
        angulos = "obtusângulo"

    return f"{lados} e {angulos}"


# A versão nova: antes de classificar, testa o alinhamento com o determinante
def classificar_triangulo_v2(A: Ponto, B: Ponto, C: Ponto) -> str:
    """Classifica o triângulo ABC, avisando quando os pontos estão alinhados."""
    D = determinante_dos_pontos(A, B, C)
    if math.isclose(D, 0, abs_tol=1e-9):
        return "os pontos estão alinhados: não formam triângulo"
    return classificar_triangulo(A, B, C)


TESTES = [((0, 0), (4, 0), (1, 3)), ((1, 1), (3, 5), (4, 7)), ((0, 5), (2, 5), (7, 5))]
for A, B, C in TESTES:
    print(f"A = {A}, B = {B}, C = {C}")
    print(f"   versão do notebook 01: {classificar_triangulo(A, B, C)}")
    print(f"   versão nova:           {classificar_triangulo_v2(A, B, C)}")

A = (0, 0), B = (4, 0), C = (1, 3)
   versão do notebook 01: escaleno e acutângulo
   versão nova:           escaleno e acutângulo
A = (1, 1), B = (3, 5), C = (4, 7)
   versão do notebook 01: escaleno e obtusângulo
   versão nova:           os pontos estão alinhados: não formam triângulo
A = (0, 5), B = (2, 5), C = (7, 5)
   versão do notebook 01: escaleno e obtusângulo
   versão nova:           os pontos estão alinhados: não formam triângulo


Repare no segundo e no terceiro casos: a função do notebook 01 "classificava" pontos alinhados como se fossem um triângulo obtusângulo (o "ângulo" do meio vale 180°). A versão nova não cai nessa armadilha.

**Pare e pense**

1. Os pontos $(0, 0)$, $(1,\ 1\,000\,000)$ e $(2,\ 2\,000\,001)$ estão alinhados? Num desenho na escala normal, dá para perceber?
2. Por que, no código, comparamos $D$ com zero usando `math.isclose` e uma tolerância, e não com `D == 0`?

<details>
<summary>Ver respostas</summary>

1. Não: $D = 0\cdot(\dots) + 1\cdot(2\,000\,001 - 0) + 2\cdot(0 - 1\,000\,000) = 1 \ne 0$. Num desenho, os três pareceriam perfeitamente alinhados. A conta enxerga o que o olho não enxerga.
2. Com coordenadas decimais, o computador arredonda as contas, e um $D$ que deveria ser 0 pode sair como `0.000000000000002`. A tolerância diz "perto o suficiente de zero conta como zero".
</details>

## 5. Abrindo a caixa-preta: a fórmula do cadarço

No módulo 04 (📎 [`0419a_equivalencia_plana.ipynb`](../04_geometria_plana/0419a_equivalencia_plana.ipynb), seção 1) você usou a "régua de área" `area_cadarco` para conferir a área de polígonos dados por coordenadas, com a promessa de que a explicação viria na Geometria Analítica. Em [`0420a_areas_de_superficies_planas.ipynb`](../04_geometria_plana/0420a_areas_de_superficies_planas.ipynb) (seção 3) você chegou perto: viu que dá para somar os triângulos de um leque **com sinal** e leu, numa frase, que essa soma "chega exatamente à fórmula do cadarço". Faltava mostrar **por que** os sinais funcionam e **como** a soma vira a fórmula. Chegou a hora.

### Passo 1: dividir em triângulos a partir de um vértice

Em [`0411a_poligonos.ipynb`](../04_geometria_plana/0411a_poligonos.ipynb) você viu que um polígono de $n$ lados pode ser dividido em $n - 2$ triângulos ligando um vértice a todos os outros, como um **leque**. Para o polígono $V_1 V_2 \dots V_n$, os triângulos são:

$$V_1 V_2 V_3,\quad V_1 V_3 V_4,\quad \dots,\quad V_1 V_{n-1} V_n$$

A área do polígono é a soma das áreas desses triângulos.

### Passo 2: somar áreas **com sinal**

Aqui entra o sinal da seção 3. Se o polígono for **convexo** e os vértices estiverem em ordem anti-horária, todos os triângulos do leque são percorridos no sentido anti-horário, e todas as áreas orientadas são positivas. Somá-las dá a área.

Mas e se o polígono tiver um "dente" (não for convexo; em [`0411a_poligonos.ipynb`](../04_geometria_plana/0411a_poligonos.ipynb), um polígono **côncavo**)? Aí alguns triângulos do leque **saem para fora** do polígono. A explicação do que acontece está na seção 3: quem está no vértice de partida vê o lado $V_k V_{k+1}$ "andar para trás" no dente, e por isso esse triângulo é percorrido no sentido **horário**. A área orientada dele é **negativa**, e ela **desconta** exatamente o pedaço que estava sobrando.

🕹️ Escolha um polígono e um vértice de partida e acrescente os triângulos do leque um por um. Observe a soma.

In [10]:
POLIGONOS = {
    "Quadrilátero convexo": [(0, 0), (6, 0), (5, 4), (1, 5)],
    "Polígono em L": [(0, 0), (6, 0), (6, 2), (2, 2), (2, 5), (0, 5)],
    "Ponta de flecha": [(0, 0), (4, 2), (0, 4), (1, 2)],
}
# A cor de cada triângulo é escolhida AQUI, pelo sinal da área orientada
COR_POSITIVA = uga.CORES["plano_secundario"]
COR_NEGATIVA = uga.CORES["destaque"]


def com_sinal(valor: float) -> str:
    """Escreve um número sempre com o sinal na frente: +4.0 ou −6.0."""
    return f"{valor:+.1f}".replace("-", "−")


def mostrar_leque(poligono: str, partida: int, mostrados: int) -> None:
    """Desenha o leque de triângulos com sinal a partir do vértice escolhido."""
    vertices = POLIGONOS[poligono]
    n = len(vertices)
    nomes = [f"V{i + 1}" for i in range(n)]
    inicio = (partida - 1) % n   # o slider vai até 6; num polígono menor, ele "dá a volta"
    # A mesma lista de vértices, agora começando pelo vértice de partida
    ordem = vertices[inicio:] + vertices[:inicio]

    # As áreas orientadas dos n − 2 triângulos do leque
    areas = [area_orientada(ordem[0], ordem[k], ordem[k + 1]) for k in range(1, n - 1)]
    quantos = min(mostrados, n - 2)   # o slider vai até 4; limitamos ao que existe

    fig = uga.criar_plano_cartesiano(
        titulo=f"{poligono}: leque a partir de {nomes[inicio]}",
        x_min=-1, x_max=7, y_min=-1, y_max=6, altura=450,
    )
    for k in range(quantos):
        triangulo = [ordem[0], ordem[k + 1], ordem[k + 2]]
        cor = COR_POSITIVA if areas[k] > 0 else COR_NEGATIVA
        fig = uga.desenhar_poligono(fig, triangulo, cor=cor, opacidade=0.35, espessura=1)
        fig = uga.adicionar_rotulo(fig, uga.baricentro(*triangulo), com_sinal(areas[k]), cor=cor)
    fig = uga.desenhar_poligono(
        fig, vertices, rotulos=nomes, cor=uga.CORES["texto"], preenchido=False, espessura=4
    )
    fig = uga.desenhar_ponto(fig, ordem[0], cor=uga.CORES["alcance"], tamanho=14)
    uga.mostrar(fig)

    mostradas = ", ".join(com_sinal(a) for a in areas[:quantos])
    print(f"Triângulos: {mostradas} → soma = {sum(areas[:quantos]):.1f}")
    print(f"Área do polígono (soma de todos os triângulos): {sum(areas):.1f}")


interact(
    mostrar_leque,
    poligono=widgets.Dropdown(
        options=list(POLIGONOS), value="Polígono em L", description="Polígono",
        style={"description_width": "initial"},
    ),
    partida=widgets.IntSlider(
        min=1, max=6, step=1, value=2, description="Vértice de partida",
        style={"description_width": "initial"}, continuous_update=False,
    ),
    mostrados=widgets.IntSlider(
        min=1, max=4, step=1, value=4, description="Triângulos mostrados",
        style={"description_width": "initial"}, continuous_update=False,
    ),
);

interactive(children=(Dropdown(description='Polígono', index=1, options=('Quadrilátero convexo', 'Polígono em …

### Passo 3: a fórmula

Ainda há um truque para deixar a conta mais simples: em vez de partir de um vértice do polígono, faça o leque a partir da **origem** $O$. Cada lado $V_i V_{i+1}$ forma, com $O$, um triângulo $O\,V_i\,V_{i+1}$, e a área orientada dele é (seção 1, com o vértice na origem):

$$\frac{1}{2}\,(x_i\, y_{i+1} - x_{i+1}\, y_i)$$

A origem pode estar dentro ou fora do polígono, tanto faz: os sinais cuidam de tudo, como você viu no widget (lá, o vértice de partida também não fazia diferença). Somando um triângulo para cada lado:

> 📌 **Fórmula do cadarço** (também conhecida como **fórmula de Gauss**). Para o polígono de vértices $V_1(x_1, y_1), \dots, V_n(x_n, y_n)$, listados em ordem ao longo do contorno:
> $$S = \frac{1}{2}\left|\sum_{i=1}^{n} (x_i\, y_{i+1} - x_{i+1}\, y_i)\right| \qquad \text{com } V_{n+1} = V_1$$

**Por que "cadarço"?** Escreva as coordenadas em duas colunas, repetindo o primeiro vértice no fim. Os produtos $x_i\, y_{i+1}$ (que entram com sinal $+$) e $x_{i+1}\, y_i$ (que entram com sinal $-$) se cruzam entre as linhas como os cordões de um tênis:

| | $x$ | $y$ |
|---|---|---|
| $V_1$ | $x_1$ | $y_1$ |
| $V_2$ | $x_2$ | $y_2$ |
| $\vdots$ | $\vdots$ | $\vdots$ |
| $V_n$ | $x_n$ | $y_n$ |
| $V_1$ | $x_1$ | $y_1$ |

> ⚠️ **Erro comum.** Listar os vértices fora de ordem. A fórmula precisa dos vértices **na ordem do contorno** (horária ou anti-horária, tanto faz). Se você pular de um lado para o outro do polígono, os "lados" da conta se cruzam e o resultado fica errado. Você vai ver isso acontecer no Exercício Desafio 9.

> 📎 **Conexão.** Para três vértices, a fórmula do cadarço é exatamente o $\frac{|D|}{2}$ da seção 2. O cadarço é a generalização do determinante dos três pontos. E é exatamente a conta que a `area_cadarco` do módulo 04 fazia em segredo: a caixa-preta está aberta.

In [11]:
def area_poligono(vertices: list) -> float:
    """Área de um polígono pela fórmula do cadarço.
    Os vértices devem estar em ordem, seguindo o contorno."""
    n = len(vertices)
    soma = 0
    for i in range(n):
        xi, yi = vertices[i]
        xj, yj = vertices[(i + 1) % n]   # o % n faz o último vértice "voltar" ao primeiro
        soma = soma + (xi * yj - xj * yi)
    return abs(soma) / 2


# Como funciona o "% n" (o resto da divisão por n): com n = 4, os índices são 0, 1, 2, 3.
# Para i = 3 (o último), o "próximo" seria 4, que não existe; mas (3 + 1) % 4 == 0,
# e o próximo vira o primeiro vértice. Confira:
print([(i + 1) % 4 for i in range(4)])   # [1, 2, 3, 0]
print()

# A tabela do cadarço do quadrilátero convexo
quadrilatero = POLIGONOS["Quadrilátero convexo"]
n = len(quadrilatero)
print(f"{'i':>2} | {'(x_i, y_i)':>10} | {'(x_i+1, y_i+1)':>14} | {'x_i·y_i+1':>9} | "
      f"{'x_i+1·y_i':>9} | {'diferença':>9}")
soma = 0
for i in range(n):
    xi, yi = quadrilatero[i]
    xj, yj = quadrilatero[(i + 1) % n]
    diferenca = xi * yj - xj * yi
    soma = soma + diferenca
    print(f"{i + 1:>2} | {(xi, yi)!s:>10} | {(xj, yj)!s:>14} | {xi * yj:>9} | "
          f"{xj * yi:>9} | {diferenca:>9}")
print(f"soma = {soma} → S = {soma}/2 = {abs(soma) / 2}")
print()

# Os três polígonos do widget, e um triângulo comparado com o determinante
for nome, vertices in POLIGONOS.items():
    print(f"{nome}: S = {area_poligono(vertices)}")
triangulo = [(1, 2), (5, 3), (2, 7)]
print(f"Triângulo {triangulo}: cadarço = {area_poligono(triangulo)}, "
      f"|D|/2 = {abs(determinante_dos_pontos(*triangulo)) / 2}")

[1, 2, 3, 0]

 i | (x_i, y_i) | (x_i+1, y_i+1) | x_i·y_i+1 | x_i+1·y_i | diferença
 1 |     (0, 0) |         (6, 0) |         0 |         0 |         0
 2 |     (6, 0) |         (5, 4) |        24 |         0 |        24
 3 |     (5, 4) |         (1, 5) |        25 |         4 |        21
 4 |     (1, 5) |         (0, 0) |         0 |         0 |         0
soma = 45 → S = 45/2 = 22.5

Quadrilátero convexo: S = 22.5
Polígono em L: S = 18.0
Ponta de flecha: S = 6.0
Triângulo [(1, 2), (5, 3), (2, 7)]: cadarço = 9.5, |D|/2 = 9.5


### De volta ao terreno do começo

Agora você tem o programa que o técnico usa. Rode a célula abaixo.

In [12]:
area_do_terreno = area_poligono(TERRENO)
# 1 hectare = 10 000 m², a área de um quadrado de 100 m de lado
hectares = area_do_terreno / 10_000
print(f"Área do terreno: {area_do_terreno} m² = {hectares} hectares")


def escrever_milhares(valor: float) -> str:
    """Escreve 22400 como "22 400", com espaço separando os milhares."""
    return f"{valor:,.0f}".replace(",", " ")


fig = desenhar_terreno(
    f"Área: {escrever_milhares(area_do_terreno)} m² ({hectares:g} ha)".replace(".", ",")
)
# O leque a partir de V₁, com a área de cada triângulo escrita no baricentro dele
cores_alternadas = [uga.CORES["plano"], uga.CORES["plano_secundario"]]
for k in range(1, len(TERRENO) - 1):
    triangulo = [TERRENO[0], TERRENO[k], TERRENO[k + 1]]
    area = area_orientada(*triangulo)
    cor = cores_alternadas[k % 2]
    fig = uga.desenhar_poligono(fig, triangulo, cor=cor, opacidade=0.3, espessura=1)
    fig = uga.adicionar_rotulo(fig, uga.baricentro(*triangulo), f"{escrever_milhares(area)} m²",
                               cor=cor, tamanho=13)
uga.mostrar(fig)

Área do terreno: 22400.0 m² = 2.24 hectares


**Pare e pense**

1. Se você listar os vértices do terreno no sentido contrário ($V_6, V_5, \dots, V_1$), o que acontece com a soma dentro do módulo? E com a área?
2. A fórmula do cadarço funciona para um polígono cujos lados se cruzam, como uma "gravata-borboleta"?

<details>
<summary>Ver respostas</summary>

1. A soma troca de sinal (o percurso passa de anti-horário a horário), e o módulo deixa a área igual.
2. Não. Num polígono que se cruza, partes da figura são percorridas no sentido horário e outras no anti-horário, e as áreas se descontam umas das outras. A fórmula vale para polígonos **simples** (sem lados que se cruzam).
</details>

### Bônus: a altura de um triângulo, de graça

A área de um triângulo também é $S = \frac{\text{base} \cdot \text{altura}}{2}$. Se você já sabe $S$ (pelo determinante) e a base $AB$ (pela distância do notebook 01), a altura relativa a $AB$ sai de uma divisão:

$$h_C = \frac{2S}{d_{AB}} = \frac{|D|}{d_{AB}}$$

**Exemplo.** $A(0, 0)$, $B(4, 3)$, $C(-1, 7)$: $D = 31$ e $d_{AB} = 5$, então $h_C = \frac{31}{5} = 6{,}2$.

> 📎 **Conexão.** Essa altura é a **distância do ponto $C$ até a reta $AB$**. No notebook 06 você vai encontrar uma fórmula direta para ela, e vai perceber que é esta mesma conta, escrita de outro jeito.

In [13]:
A5, B5, C5 = (0, 0), (4, 3), (-1, 7)

# As contas: D pelo determinante, a base pela distância (notebook 01), a altura por divisão
D5 = determinante_dos_pontos(A5, B5, C5)
base = uga.distancia(A5, B5)
altura = abs(D5) / base
print(f"D = {D5}, d(A, B) = {base}, h = |D| / d(A, B) = {abs(D5)} / {base:g} = {altura}")


def pe_da_altura(A: Ponto, B: Ponto, C: Ponto) -> tuple:
    """O ponto H da reta AB em que cai a altura que sai de C (usado só para desenhar)."""
    # Esta conta (projeção) será explicada no notebook 03. Aqui ela só serve para desenhar o pé da altura.
    t = ((C[0] - A[0]) * (B[0] - A[0]) + (C[1] - A[1]) * (B[1] - A[1])) / uga.distancia(A, B)**2
    return uga.ponto_do_segmento(A, B, t)


H = pe_da_altura(A5, B5, C5)
fig = uga.criar_plano_cartesiano(
    titulo="A altura relativa ao lado AB", x_min=-4, x_max=8, y_min=-2, y_max=9
)
fig = uga.desenhar_reta_por_dois_pontos(fig, A5, B5)
fig = uga.desenhar_poligono(fig, [A5, B5, C5], rotulos=["A", "B", "C"])
fig = uga.desenhar_segmento(
    fig, C5, H, rotulo=f"h = {altura:g}".replace(".", ","), cor=uga.CORES["destaque"],
    tracejado=True,
)
fig = uga.desenhar_ponto(fig, H, rotulo="H", cor=uga.CORES["destaque"], tamanho=7,
                         posicao_rotulo="bottom right")
uga.mostrar(fig)

D = 31, d(A, B) = 5.0, h = |D| / d(A, B) = 31 / 5 = 6.2


## Resumo

**Determinante dos três pontos**
$$D = \begin{vmatrix} x_A & y_A & 1 \\ x_B & y_B & 1 \\ x_C & y_C & 1 \end{vmatrix} = \begin{vmatrix} x_B - x_A & y_B - y_A \\ x_C - x_A & y_C - y_A \end{vmatrix} = x_A(y_B - y_C) + x_B(y_C - y_A) + x_C(y_A - y_B)$$

| O que você quer | Como obter |
|---|---|
| Área do triângulo | $S = \frac{\lvert D \rvert}{2}$ |
| Sentido de $A \to B \to C$ | $D > 0$: anti-horário · $D < 0$: horário |
| De que lado de $AB$ está $C$ | $D > 0$: à esquerda de quem anda de $A$ para $B$ · $D < 0$: à direita |
| $A$, $B$, $C$ alinhados? | $D = 0$ |
| Altura relativa a $AB$ | $h_C = \frac{\lvert D \rvert}{d_{AB}}$ |
| Área de um polígono (vértices em ordem) | $S = \frac{1}{2}\left\lvert \sum (x_i\, y_{i+1} - x_{i+1}\, y_i) \right\rvert$ (cadarço) |

**Lembretes**
- Não esqueça de dividir por 2 e de tirar o módulo.
- Trocar dois vértices de lugar troca o sinal de $D$, mas não a área.
- No cadarço, os vértices precisam estar na ordem do contorno, e o polígono não pode se cruzar.
- No computador, teste $D = 0$ com tolerância (`math.isclose`).

## Exercícios

Responda preenchendo as variáveis indicadas em cada célula (troque cada `...` pela sua resposta) e rode a célula de verificação logo abaixo. Os exercícios estão em três níveis: **básico**, **intermediário** e **desafio**. Você pode usar as funções que programou neste notebook (`determinante_dos_pontos`, `determinante_dos_deslocamentos`, `area_orientada`, `area_poligono`) e as ferramentas do notebook 01 que estão no `uga` (`uga.distancia`, `uga.ponto_medio` etc.).

O gabarito comentado está em [`0702b_alinhamento_e_areas_por_determinantes_gabarito.ipynb`](0702b_alinhamento_e_areas_por_determinantes_gabarito.ipynb). Tente sozinho antes de olhar!

### Básico

In [ ]:
# Exercício Básico 1: área. Calcule a área do triângulo A(1, 2), B(5, 3), C(2, 7).
area_ex1 = ...

print(area_ex1)

In [ ]:
# Verificação
assert area_ex1 is not ... and math.isclose(area_ex1, 9.5), (
    "Tente de novo, revise: calcule D e lembre de dividir por 2 "
    "(seção 2 - O determinante dos três pontos)."
)
print("Certo!")

In [ ]:
# Exercício Básico 2: alinhados ou não? Responda True ou False para cada trio de pontos.
# (a) (1, 1), (3, 5), (4, 7);
# (b) (-2, 3), (1, 1), (4, -2);
# (c) (0, 5), (2, 5), (7, 5).
alinhados_a_ex2 = ...
alinhados_b_ex2 = ...
alinhados_c_ex2 = ...

print(alinhados_a_ex2, alinhados_b_ex2, alinhados_c_ex2)

In [ ]:
# Verificação
respostas_ex2 = [
    ("a", alinhados_a_ex2, True), ("b", alinhados_b_ex2, False), ("c", alinhados_c_ex2, True),
]
for item, resposta, certa in respostas_ex2:
    assert resposta is not ... and resposta == certa, (
        f"Tente de novo, revise: no item ({item}), calcule D; os pontos estão alinhados "
        "exatamente quando D = 0 (seção 4 - Três pontos alinhados)."
    )
print("Certo!")

In [ ]:
# Exercício Básico 3: sentido. Para A(0, 0), B(4, 0) e C(1, 3), responda com o texto
# "horario" ou "anti-horario" (sem acento):
# (a) o sentido do percurso A → B → C;
# (b) o sentido do percurso A → C → B.
sentido_a_ex3 = ...
sentido_b_ex3 = ...

print(sentido_a_ex3, sentido_b_ex3)

In [ ]:
# Verificação
def normalizar_sentido(resposta) -> str:
    """Aceita a resposta com ou sem acento, com letras maiúsculas ou espaços sobrando."""
    if not isinstance(resposta, str):
        return ""
    return resposta.strip().lower().replace("á", "a")


respostas_ex3 = [("a", sentido_a_ex3, "anti-horario"), ("b", sentido_b_ex3, "horario")]
for item, resposta, certa in respostas_ex3:
    assert normalizar_sentido(resposta) == certa, (
        f"Tente de novo, revise: no item ({item}), D > 0 é anti-horário e D < 0 é horário; "
        "trocar dois vértices troca o sinal (seção 3 - O sinal de D: horário ou anti-horário?)."
    )
print("Certo!")

### Intermediário

In [ ]:
# Exercício Intermediário 4: encontre k. Para que valor de k os pontos (k, 4), (1, 2)
# e (3, 6) estão alinhados?
k_ex4 = ...

print(k_ex4)

In [ ]:
# Verificação
assert k_ex4 is not ... and math.isclose(k_ex4, 2), (
    "Tente de novo, revise: monte D com k no lugar de uma coordenada, simplifique e "
    "resolva D = 0 (seção 4 - Três pontos alinhados)."
)
print("Certo!")

In [ ]:
# Exercício Intermediário 5: quadrilátero. Calcule a área do quadrilátero de vértices,
# nesta ordem, (-2, -1), (4, -2), (6, 3) e (0, 5).
area_ex5 = ...

print(area_ex5)

In [ ]:
# Verificação
assert area_ex5 is not ... and math.isclose(area_ex5, 36), (
    "Tente de novo, revise: monte a tabela do cadarço, repetindo o primeiro vértice no "
    "fim (seção 5 - Abrindo a caixa-preta: a fórmula do cadarço)."
)
print("Certo!")

In [ ]:
# Exercício Intermediário 6: altura. No triângulo A(1, 1), B(7, 9), C(8, 2), calcule a
# altura relativa ao lado AB.
altura_ex6 = ...

print(altura_ex6)

In [ ]:
# Verificação
assert altura_ex6 is not ... and math.isclose(altura_ex6, 5), (
    "Tente de novo, revise: h = |D| / d(A, B); cuidado com o módulo "
    "(seção 5 - Bônus: a altura de um triângulo, de graça)."
)
print("Certo!")

In [ ]:
# Exercício Intermediário 7: área dada, vértice desconhecido. O triângulo A(0, 0),
# B(4, 0), C(k, k) tem área 10. Encontre os DOIS valores possíveis de k e guarde-os
# numa lista, em qualquer ordem, por exemplo [1, 2].
valores_k_ex7 = ...

print(valores_k_ex7)

In [ ]:
# Verificação
def mesmos_numeros(resposta, certos: list) -> bool:
    """Confere se a resposta é uma lista com os números certos, em qualquer ordem."""
    try:
        return len(resposta) == len(certos) and all(
            math.isclose(r, c, abs_tol=1e-9) for r, c in zip(sorted(resposta), sorted(certos))
        )
    except TypeError:   # a resposta não é uma lista de números (ou ainda é ...)
        return False


assert mesmos_numeros(valores_k_ex7, [-5, 5]), (
    "Tente de novo, revise: o módulo deixa duas possibilidades: 2k = 10 ou 2k = −10. "
    "Desenhe os dois triângulos! (seção 2 - O determinante dos três pontos)."
)
print("Certo!")

Depois de acertar, rode a célula abaixo para ver os dois triângulos.

In [ ]:
# Verificação visual do Exercício Intermediário 7 (rode depois de acertar).
assert mesmos_numeros(valores_k_ex7, [-5, 5]), (
    "Tente de novo, revise: primeiro resolva o exercício acima "
    "(seção 2 - O determinante dos três pontos)."
)
fig = uga.criar_plano_cartesiano(
    titulo="Os dois triângulos de área 10", x_min=-7, x_max=7, y_min=-7, y_max=7
)
for k, cor in zip(sorted(valores_k_ex7), [COR_HORARIO, COR_ANTI_HORARIO]):
    fig = uga.desenhar_poligono(fig, [(0, 0), (4, 0), (k, k)], cor=cor)
    fig = uga.desenhar_ponto(fig, (k, k), rotulo="C", cor=cor, mostrar_coordenadas=True)
fig = uga.desenhar_pontos(fig, [(0, 0), (4, 0)], rotulos=["A", "B"])
uga.mostrar(fig)
print("Um triângulo fica acima do eixo x (D > 0) e o outro, abaixo (D < 0).")

### Desafio

In [ ]:
# Exercício Desafio 8: ponto dentro do triângulo. Complete a função
# ponto_no_triangulo(P, A, B, C): ela deve devolver True se P está dentro do triângulo
# ABC OU NA BORDA, e False se está fora. Use apenas sinais de determinantes.
# Ideia: calcule D(A, B, P), D(B, C, P) e D(C, A, P). P está dentro (ou na borda)
# quando nenhum deles é positivo OU nenhum deles é negativo.
# A função deve funcionar com os vértices em qualquer sentido (horário ou anti-horário).
def ponto_no_triangulo(P: Ponto, A: Ponto, B: Ponto, C: Ponto) -> bool:
    ...  # complete aqui


print(ponto_no_triangulo((2, 1), (0, 0), (4, 0), (1, 3)))

In [ ]:
# Verificação
casos_ex8 = [((2, 1), True), ((4, 3), False), ((1, 0), True), ((-1, 0), False)]
for vertices, sentido in [([(0, 0), (4, 0), (1, 3)], "anti-horário"),
                          ([(0, 0), (1, 3), (4, 0)], "horário")]:
    for P, esperado in casos_ex8:
        assert ponto_no_triangulo(P, *vertices) == esperado, (
            f"Tente de novo, revise: com os vértices {vertices} (sentido {sentido}), "
            f"P{P} deveria dar {esperado}; P está dentro quando os três D não têm sinais "
            "contrários (nenhum positivo ou nenhum negativo) "
            "(seção 3 - O sinal de D: horário ou anti-horário?)."
        )
print("Certo!")

Depois de acertar, rode a célula abaixo e mova o ponto $P$: ele fica verde dentro do triângulo e vermelho fora, decidido pela **sua** função.

In [ ]:
# Verificação visual do Exercício Desafio 8 (rode depois de acertar).
TRIANGULO_EX8 = [(0, 0), (4, 0), (1, 3)]


def mostrar_ponto_no_triangulo(xP: float, yP: float) -> None:
    """Pinta P de verde (dentro ou na borda) ou de vermelho (fora), com a sua função."""
    P = (xP, yP)
    dentro = ponto_no_triangulo(P, *TRIANGULO_EX8)
    cor = uga.CORES["plano_secundario"] if dentro else uga.CORES["destaque"]
    fig = uga.criar_plano_cartesiano(
        titulo="Dentro ou fora?", x_min=-2, x_max=5, y_min=-1, y_max=4, altura=400
    )
    fig = uga.desenhar_poligono(fig, TRIANGULO_EX8, rotulos=["A", "B", "C"])
    fig = uga.desenhar_ponto(fig, P, rotulo="P", cor=cor, tamanho=12, mostrar_coordenadas=True)
    uga.mostrar(fig)
    print("P está dentro (ou na borda)." if dentro else "P está fora.")


interact(
    mostrar_ponto_no_triangulo,
    xP=widgets.FloatSlider(min=-2, max=5, step=0.25, value=2, description="x_P",
                           continuous_update=False),
    yP=widgets.FloatSlider(min=-1, max=4, step=0.25, value=1, description="y_P",
                           continuous_update=False),
);

In [ ]:
# Exercício Desafio 9: o "L" e a ordem dos vértices.
# (a) Calcule a área do polígono em "L" de vértices (0, 0), (6, 0), (6, 2), (2, 2),
#     (2, 5), (0, 5) de DOIS jeitos: pela fórmula do cadarço (area_cadarco_ex9) e
#     dividindo o L em dois retângulos (area_retangulos_ex9).
# (b) Agora use a fórmula do cadarço com os mesmos vértices na ordem
#     (0, 0), (6, 0), (2, 2), (6, 2), (2, 5), (0, 5) e guarde o resultado em
#     area_ordem_errada_ex9.
area_cadarco_ex9 = ...
area_retangulos_ex9 = ...
area_ordem_errada_ex9 = ...

print(area_cadarco_ex9, area_retangulos_ex9, area_ordem_errada_ex9)

In [ ]:
# Verificação
assert area_cadarco_ex9 is not ... and math.isclose(area_cadarco_ex9, 18), (
    "Tente de novo, revise: no item (a), aplique o cadarço aos vértices na ordem dada "
    "(seção 5 - Abrindo a caixa-preta: a fórmula do cadarço)."
)
assert area_retangulos_ex9 is not ... and math.isclose(area_retangulos_ex9, 18), (
    "Tente de novo, revise: no item (a), corte o L num retângulo 6 × 2 e num 2 × 3 "
    "(seção 5 - Abrindo a caixa-preta: a fórmula do cadarço)."
)
assert area_ordem_errada_ex9 is not ... and math.isclose(area_ordem_errada_ex9, 20), (
    "Tente de novo, revise: no item (b), use area_poligono com os vértices exatamente na "
    "nova ordem (seção 5 - Abrindo a caixa-preta: a fórmula do cadarço)."
)
print("Certo!")

**(b) continuação.** A área do item (b) deu diferente de 18. Por que deu errado? Escreva a sua explicação na célula de texto abaixo (dê dois cliques nela para editar). Depois, rode a célula de código seguinte para ver os dois contornos e conferir a sua ideia.

*Escreva aqui a sua explicação do item (b).*

In [ ]:
# Verificação visual do Exercício Desafio 9 (rode depois de responder).
L_CERTO = [(0, 0), (6, 0), (6, 2), (2, 2), (2, 5), (0, 5)]
L_ERRADO = [(0, 0), (6, 0), (2, 2), (6, 2), (2, 5), (0, 5)]
for vertices, titulo in [(L_CERTO, "Ordem certa"), (L_ERRADO, "Ordem errada")]:
    fig = uga.criar_plano_cartesiano(
        titulo=f"{titulo}: cadarço = {area_poligono(vertices):g}",
        x_min=-1, x_max=7, y_min=-1, y_max=6, altura=380,
    )
    fig = uga.desenhar_poligono(fig, vertices, rotulos=[f"V{i + 1}" for i in range(6)])
    fig = uga.desenhar_sentido_de_percurso(fig, vertices)
    uga.mostrar(fig)

In [ ]:
# Exercício Desafio 10: conferindo o Teorema de Pick.
# No Desafio 7 de 0420a_areas_de_superficies_planas (módulo 04) você conheceu o Teorema
# de Pick: se um polígono tem todos os vértices em pontos de coordenadas inteiras,
#     S = I + B/2 − 1,
# em que I é o número de pontos inteiros DENTRO do polígono e B é o número de pontos
# inteiros NA BORDA (vértices incluídos). Lá, B veio do mdc e I de uma semirreta.
# Agora, use SÓ DETERMINANTES.
# Complete a função pick_por_determinantes(vertices), para um polígono CONVEXO com os
# vértices em ordem ANTI-HORÁRIA. Ela deve devolver a tupla (S, B, I):
#   1. S pela fórmula do cadarço;
#   2. B e I percorrendo todos os pontos inteiros P do retângulo envolvente:
#      - P está NA BORDA se está alinhado com algum lado Vi Vi+1 (determinante zero)
#        E dentro do retângulo envolvente desse lado;
#      - P está DENTRO se, para todos os lados, D(Vi, Vi+1, P) > 0.
# Teste com o triângulo (0, 0), (4, 1), (1, 3) e o quadrilátero (1, 0), (7, 2), (4, 6), (0, 4),
# e confira se S = I + B/2 − 1 nos dois.
def pick_por_determinantes(vertices: list) -> tuple:
    ...  # complete aqui


print(pick_por_determinantes([(0, 0), (4, 1), (1, 3)]))
print(pick_por_determinantes([(1, 0), (7, 2), (4, 6), (0, 4)]))

In [ ]:
# Verificação
casos_ex10 = [
    ([(0, 0), (4, 1), (1, 3)], (5.5, 3, 5)),
    ([(1, 0), (7, 2), (4, 6), (0, 4)], (24, 6, 22)),
    ([(0, 0), (4, 0), (4, 4), (0, 4)], (16, 16, 9)),   # um quadrado, para controle
]
for vertices, (S_certo, B_certo, I_certo) in casos_ex10:
    resposta = pick_por_determinantes(vertices)
    assert resposta is not None and len(resposta) == 3, (
        "Tente de novo, revise: a função deve devolver a tupla (S, B, I) "
        "(seção 5 - Abrindo a caixa-preta: a fórmula do cadarço)."
    )
    S, B, I = resposta
    assert math.isclose(S, S_certo), (
        f"Tente de novo, revise: para {vertices}, S deveria ser {S_certo}; use o cadarço "
        "(seção 5 - Abrindo a caixa-preta: a fórmula do cadarço)."
    )
    assert B == B_certo, (
        f"Tente de novo, revise: para {vertices}, cuidado para não contar duas vezes os "
        "vértices (eles estão em dois lados ao mesmo tempo) (seção 4 - Três pontos alinhados)."
    )
    assert I == I_certo, (
        f"Tente de novo, revise: para {vertices}, P está dentro quando D(Vi, Vi+1, P) > 0 "
        "para TODOS os lados (seção 3 - O sinal de D: horário ou anti-horário?)."
    )
    assert math.isclose(S, I + B / 2 - 1), (
        "Tente de novo, revise: confira as contagens de B e I "
        "(seção 4 - Três pontos alinhados)."
    )
print("Certo!")

## Conexões

**Isso depende de:**
- [`0701a_plano_cartesiano_ponto_e_distancia.ipynb`](0701a_plano_cartesiano_ponto_e_distancia.ipynb) (deslocamentos, distância e a função `classificar_triangulo`)
- [`0305a_determinantes.ipynb`](../03_progressoes_matrizes_e_sistemas_lineares/0305a_determinantes.ipynb) (determinantes, suas propriedades e o paralelogramo com sinal)
- [`0401a_nocoes_e_proposicoes_primitivas.ipynb`](../04_geometria_plana/0401a_nocoes_e_proposicoes_primitivas.ipynb) (o teste de colinearidade multiplicando em cruz) e [`0408a_quadrilateros_definicao_elementos_e_trapezios.ipynb`](../04_geometria_plana/0408a_quadrilateros_definicao_elementos_e_trapezios.ipynb) (a função `orientacao`)
- [`0409a_paralelogramos_e_casos_particulares.ipynb`](../04_geometria_plana/0409a_paralelogramos_e_casos_particulares.ipynb) (a diagonal divide o paralelogramo em dois triângulos congruentes)
- [`0411a_poligonos.ipynb`](../04_geometria_plana/0411a_poligonos.ipynb) (divisão de um polígono em $n - 2$ triângulos)
- [`0419a_equivalencia_plana.ipynb`](../04_geometria_plana/0419a_equivalencia_plana.ipynb) (mesma base e mesma altura; a régua de área `area_cadarco`)
- [`0420a_areas_de_superficies_planas.ipynb`](../04_geometria_plana/0420a_areas_de_superficies_planas.ipynb) (a área do triângulo pelas coordenadas, o leque com sinal e o Teorema de Pick)

**Isso será usado em:**
- 📌 Vetores no plano (notebook 03: os deslocamentos viram vetores, e o determinante 2 × 2 ganha um nome próprio na geometria)
- 📌 Equações da reta (notebook 04: a condição de alinhamento com um ponto genérico $P(x, y)$ vira a equação da reta)
- 📌 Distância de ponto a reta e aplicações (notebook 06: a altura $\frac{|D|}{d_{AB}}$ reaparece como fórmula)
- 📌 Inequações, regiões e programação linear (notebook 07: o "lado da reta" vira um semiplano)
- 📌 Produto vetorial e produto misto (notebook 18: a versão espacial do determinante como área e volume)

Para revisar os termos do módulo, use o [glossário de Geometria Analítica](glossario.md).